# Machine Learning for Membrane Development

Welcome. This Google Colab / Jupyter notebook walks through a full machine-learning (ML) workflow on **membrane** data: how a film is made, how well water and salts go through it, and how we might suggest a better recipe.

You only need **basic Python** (variables, functions, running cells in order). You do **not** need to know membrane chemistry or Bayesian optimization before you start.

## Learning objectives

By the end you should be able to:

1. Tell **input features** (what we know about a recipe) from **targets** (the numbers we want to predict).
2. Load a table and spot missing values and duplicate rows.
3. Add chemistry-informed columns (ion properties) before training.
4. Train a tree model, split **train** vs **test**, and read a parity plot.
5. Use SHAP plots as *hypotheses*, not as proof of a physical mechanism.
6. Run a short **inverse design** search (Ax) for **continuous** targets only.

**Important limits.** Supervised models here can do **regression** (continuous KPIs such as permeability) or **classification** (labels such as pass/fail). The Ax search in §6 is **regression only**. A good test score does **not** mean the model discovered the true physics.

Run cells **from top to bottom**. On Colab, do the bootstrap cell in the next section **before** Section 0.


### Prerequisites and setup (Google Colab)

**What you will do.** Run the next cell once so Colab has the full repository and Python packages.

**Why it matters.** Opening only this `.ipynb` from GitHub does **not** download `config.yaml`, `src/`, or the data files. Bootstrap clones (or updates) `ML-MEM-Review` and runs `pip install -e ".[all]"`.

**What to look for.** Printed `Repo root: /content/ML-MEM-Review` and `Colab bootstrap done.` The first install can take **several minutes** (PyTorch / Ax / SHAP). You do **not** need Google Drive.

**Common pitfall.** Skipping bootstrap, then later cells fail with `config.yaml` or `No module named ...`. Fix: run this cell, then Section 0. After a GitHub update, run bootstrap again (it refreshes the clone).

On a **local** computer the cell mostly enables plots; install once with `pip install -e ".[all]"` from the repo root.

Details: [`docs/COLAB.md`](../../docs/COLAB.md).


In [ ]:
# @title Bootstrap (clone + install)
# === COLAB BOOTSTRAP (safe locally) =========================================
# Clones Oikiy/ML-MEM-Review if you only opened the .ipynb from GitHub,
# installs the package editable, and cd's to the repo root.
import os
import sys
import subprocess
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _run(cmd, **kwargs):
    print("+", " ".join(cmd) if isinstance(cmd, (list, tuple)) else cmd)
    subprocess.check_call(cmd, **kwargs)

def _find_repo_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").is_file() and (candidate / "src").is_dir():
            return candidate
    return None

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

IN_COLAB = _in_colab()
REPO_URL = os.environ.get("ML_MEM_REPO_URL", "https://github.com/Oikiy/ML-MEM-Review.git")
REPO_NAME = os.environ.get("ML_MEM_REPO_NAME", "ML-MEM-Review")
REPO_REF = os.environ.get("ML_MEM_REPO_REF", "main")
COLAB_ROOT = Path(os.environ.get("ML_MEM_COLAB_ROOT", f"/content/{REPO_NAME}"))
EXTRAS = 'all'

def _git_head(repo: Path) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError, OSError):
        return ""

def _truthy(name: str) -> bool:
    return os.environ.get(name, "").strip().lower() in {"1", "true", "yes"}

if IN_COLAB:
    if COLAB_ROOT.exists() and (COLAB_ROOT / ".git").is_dir():
        if _truthy("ML_MEM_SKIP_PULL"):
            print(f"Skipping git update at {COLAB_ROOT} (ML_MEM_SKIP_PULL=1)")
        else:
            print(f"Updating clone at {COLAB_ROOT} to origin/{REPO_REF}")
            _run(["git", "-C", str(COLAB_ROOT), "fetch", "--depth", "1", "origin", REPO_REF])
            _run(["git", "-C", str(COLAB_ROOT), "reset", "--hard", "FETCH_HEAD"])
    elif not COLAB_ROOT.exists():
        _run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(COLAB_ROOT)])
    os.chdir(COLAB_ROOT)
    root = _find_repo_root(Path.cwd())
else:
    root = _find_repo_root(Path.cwd())

if root is None:
    raise RuntimeError(
        "Could not locate the ML-MEM-Review repo (need config.yaml + src/). "
        "On Colab: re-run this cell or set ML_MEM_REPO_URL to your fork. "
        "Locally: open the notebook from inside a full clone of the repository."
    )

os.chdir(root)
print(f"Repo root: {root}  git={_git_head(root)[:12]}")

src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB:
    marker = root / ".colab_install_ok"
    force = _truthy("ML_MEM_FORCE_INSTALL")
    head = _git_head(root)
    extras = EXTRAS
    pip_spec = str(root) if not extras else f"{root}[{extras}]"
    install_key = f"{head}\n{extras}"
    prev = marker.read_text(encoding="utf-8").strip() if marker.is_file() else ""
    if force or prev != install_key:
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", pip_spec])
        try:
            marker.write_text(install_key + "\n", encoding="utf-8")
        except OSError:
            pass
    else:
        print("Skipping pip install (clone SHA + extras unchanged). Set ML_MEM_FORCE_INSTALL=1 to reinstall.")
    print("Colab bootstrap done.")
else:
    print("Local session: repo root + src/ on sys.path (no auto pip install).")

if EXTRAS:
    try:
        import ax  # noqa: F401
    except ImportError:
        pip_spec = f"{root}[{EXTRAS}]"
        print(f"Ax not found in this kernel; installing {pip_spec}")
        _run([sys.executable, "-m", "pip", "install", "-e", pip_spec])

print("Next: run the Section 0 import cell below.")
# === end bootstrap ===========================================================


## Section 0. Setup

**What you will do.** Import libraries, load `config.yaml`, and set the random seed and figure style.

**Why it matters.** `config.yaml` is the control panel: where the data live, which column is the target, which model to use, and (later) the Ax search. Changing YAML is how you reuse this package on a new table without rewriting the tutorial.

**Useful words**

- **Repository (repo):** the project folder with `config.yaml` and `src/`.
- **Random seed:** a number that makes shuffled splits repeatable.

**What to look for.** Printed paths for the dataset, cation/anion lookup tables, `models/`, and `outputs/figures/`. On Colab those sit under `/content/ML-MEM-Review/...`.

**Common pitfall.** Running this cell before bootstrap on Colab. Ax may warn here; that is OK until Section 6.

**Try it yourself.** Before you run: which file do you think tells the notebook the Excel/CSV path — a Python variable you type, or `config.yaml`?


In [ ]:
# For file handling and data manipulation
import os
import joblib
import numpy as np
import pandas as pd
import sys
import warnings

# For YAML configuration loading
try:
    import yaml
except ImportError:
    import subprocess
    print("Installing PyYAML...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml"])
    import yaml

# For modeling and evaluation
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# For SHAP explanations (optional here; required later in SHAP section)
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# For inverse design (Ax is optional until Section 6)
try:
    from ax.api.client import Client
    from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig
except Exception as exc:
    Client = None
    RangeParameterConfig = None
    ChoiceParameterConfig = None
    warnings.warn(
        f"Ax import skipped during setup: {exc}. "
        "You can still run Sections 1-5; fix Ax/PyTorch before Section 6.",
        RuntimeWarning,
    )

# ============================================================================
# CONFIGURATION SETUP: Load all settings from config.yaml
# ============================================================================

# Navigate to project root and add src to path for utl_config_loader
PROJECT_ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(PROJECT_ROOT, "config.yaml")):
    parent = os.path.dirname(PROJECT_ROOT)
    if parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not find config.yaml above the current folder. "
            "Launch the notebook from inside the repository."
        )
    PROJECT_ROOT = parent
CODE_PATH = os.path.join(PROJECT_ROOT, "src")

if CODE_PATH not in sys.path:
    sys.path.insert(0, CODE_PATH)
    print(f"Added {CODE_PATH} to sys.path for module imports.")

# Load configuration from YAML file
from utl_config_loader import load_config, get_data_paths, get_model_dir, get_figures_dir, get_plotting_config

config = load_config(project_root=PROJECT_ROOT)
print(f"\nConfiguration loaded from: {config['_config_path']}")

# ============================================================================
# PLOTTING CONFIGURATION
# ============================================================================

plotting_cfg = config["plotting"]

# Extract and apply plotting settings
FONT_SIZE = plotting_cfg["font_size"]
FIG_SIZE = tuple(plotting_cfg["global_figure_size"])
DPI = plotting_cfg["dpi"]

# Apply matplotlib settings
plt.rcParams["font.size"] = FONT_SIZE
plt.rcParams["figure.figsize"] = FIG_SIZE
plt.rcParams["savefig.dpi"] = DPI

# Nicer plot defaults
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Ellipse, Rectangle

sns.set(style="whitegrid", context="talk")

# Create custom colormap from config
colormap_cfg = plotting_cfg["custom_colormap"]
custom_cmap = LinearSegmentedColormap.from_list(
    colormap_cfg["name"],
    colormap_cfg["colors"]
)

print(f"Plotting config applied: FIG_SIZE={FIG_SIZE}, FONT_SIZE={FONT_SIZE}, DPI={DPI}")

# ============================================================================
# MODELING CONFIGURATION
# ============================================================================

RANDOM_STATE = config["modeling"]["random_state"]
np.random.seed(RANDOM_STATE)

# ============================================================================
# DATA PATHS (from config)
# ============================================================================

data_paths = get_data_paths(config)
DATA_PATH = data_paths["dataset"]

lookup_by_align_column = data_paths.get("lookup_by_align_column", {})
CATION_PROP_PATH = lookup_by_align_column.get("S1_Cation")
ANION_PROP_PATH = lookup_by_align_column.get("S1_Anion")

if CATION_PROP_PATH is None or ANION_PROP_PATH is None:
    missing_lookup_keys = [
        key for key, value in {"S1_Cation": CATION_PROP_PATH, "S1_Anion": ANION_PROP_PATH}.items()
        if value is None
    ]
    raise KeyError(
        "Missing lookup path(s) in feature_engineering.lookups for: "
        f"{missing_lookup_keys}. Please define lookups with align_column values "
        "S1_Cation and S1_Anion in config.yaml."
    )

MODEL_PATH = get_model_dir(config)
FIGURES_DIR = get_figures_dir(config)

# Ensure output directories exist
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Data paths configured:")
print(f"  - Tutorial dataset: {os.path.basename(DATA_PATH)}")
print(f"  - Cation lookup: {os.path.basename(CATION_PROP_PATH)}")
print(f"  - Anion lookup: {os.path.basename(ANION_PROP_PATH)}")
print(f"  - Models directory: {MODEL_PATH}")
print(f"  - Figures output: {FIGURES_DIR}")


## Section 1. Raw dataset — load and look

**What you will do.** Read the literature table and look at its size and column names.

**Why it matters.** Every later step assumes you know what **one row** is: one published membrane experiment (recipe + measured performance), not a random independent draw from nature.

**Features vs targets (this notebook)**

- **Features (inputs):** fabrication and test conditions, e.g. monomer name, mass, polymerization time, solute identity (`S1_Cation`, `S1_Anion`).
- **Targets (outputs to predict):** e.g. pure-water permeability and solute rejection.

The bundled table is on the order of **~278 rows × ~36 columns**. If you pointed `config.yaml` at another file, your numbers will differ.

**What to look for.** A pandas preview: recognisable columns such as `DOI`, `Dope1_Monomer1_Name`, `solute1_Rejection`.

**Try it yourself.** From the column list, write down one feature and one target before you go on.


In [ ]:
# Load the dataset into a pandas DataFrame
df = pd.read_excel(DATA_PATH)

# Show the shape and first few rows of the dataset
print("Dataset shape (rows, columns):", df.shape)
df.head()

### How the columns are grouped

Think of the table as three notebooks stapled together:

1. **How the film was made** — monomers, additives, solvents, time, temperature.
2. **How it was tested** — pressure, feed, compaction when the paper reported them.
3. **What was in the water** — solute name and ions (`Solute1_Name`, `S1_Cation`, `S1_Anion`).

Section 2 will attach extra chemistry numbers to those ion names. You are still only **looking**, not cleaning.


### Missing values and duplicate rows

**What you will do.** Count missing entries (empty cells) and exact duplicate rows.

**Why it matters.** Empty process conditions shrink the usable sample. Exact duplicates can make the model look better than it is because the same paper is counted twice.

**What to look for.** A per-column missing **count and %**. A high % on a column you hoped to use is a warning, not an automatic delete.

**Common pitfall.** Dropping every row that has any blank. You can throw away most of a small materials table. In this tutorial, simple filling-in happens **later**, inside the training pipeline, and only using the **training** split (so the test set stays honest).

**Try it yourself.** If `polymerization time` is missing in 80% of rows, would you impute it, drop the column, or keep an “is missing?” flag? There is no single right answer — that is the point.


In [ ]:
# Check the data types and missing values
print("\nData types and missing values:")
print("---"*30)
print(df.info())

# Check for duplicates
num_duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {num_duplicates}")

### Optional deeper dive: why values are missing

You can skip this box and still finish the notebook. Imputation is **not** applied in the next code cell.

Statisticians name three patterns:

- **MCAR** — missing by accident (a logger dropped a point). Simple averages are often OK if few cells are empty.
- **MAR** — missingness depends on **other columns we did record**.
- **MNAR** — missing because of the missing value itself (extreme conditions never published). Blind averages can mislead.

Membrane papers also have **structured** gaps: a method was never measured. Then a domain default or dropping the column is more honest than a made-up mean.

**In this tutorial.** Section 4’s scikit-learn pipeline fills holes using **training data only**, then applies the same rule to test rows (no leakage).


In [ ]:
# Implement duplicate-row handling identified above.
num_duplicates = df.duplicated().sum()
if num_duplicates > 0:
    print(f"Dropping {num_duplicates} duplicate rows (keeping the first occurrence).")
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    print(f"Rows remaining after deduplication: {len(df)}")
else:
    print("No duplicate rows to drop.")


In [ ]:
# Load target and feature selections from config.
import utl_config_loader as cfg_loader

# Canonical target properties for modeling (KPIs)
configured_candidate_targets = cfg_loader.resolve_targets(config)

print("Configured targets from config (data.targets):")
for i, target_name in enumerate(configured_candidate_targets, start=1):
    print(f"  {i}. {target_name}")

# Keep only targets that are present in the loaded dataframe
candidate_targets = [t for t in configured_candidate_targets if t in df.columns]
missing_targets = [t for t in configured_candidate_targets if t not in df.columns]

print("\nAvailable targets in dataset:", candidate_targets)
if missing_targets:
    print("Targets listed in config but missing from dataset:", missing_targets)

if not candidate_targets:
    raise ValueError(
        "None of the configured data.targets columns were found in the dataset."
    )

# Raw columns to include for tutorial EDA/modeling subset
configured_cols_to_show = cfg_loader.resolve_features_to_model(config, df)

print("\nFeatures selected from config (features.mode in config.yaml):")
for i, col_name in enumerate(configured_cols_to_show, start=1):
    print(f"  {i}. {col_name}")

# Keep only configured columns that exist in the dataframe
available_feature_columns = [c for c in configured_cols_to_show if c in df.columns]
missing_feature_columns = [c for c in configured_cols_to_show if c not in df.columns]

if missing_feature_columns:
    print("Configured features missing from dataset:", missing_feature_columns)

if not available_feature_columns:
    raise ValueError(
        "None of the configured features were found in the dataset. Check features.mode and features.include in config.yaml."
    )

# Append available targets to the selected feature columns (avoid duplicates)
cols_to_show = available_feature_columns + [
    t for t in candidate_targets if t not in available_feature_columns
]

# Filter the DataFrame to only include those columns for modeling and EDA
df_filtered = df[cols_to_show]
display(df_filtered.head())

# Separate numeric and categorical columns for later processing
df_numeric = df_filtered.select_dtypes(include=[np.number])
df_categorical = df_filtered.select_dtypes(include=["object"])

## Section 2. Feature engineering

**What you will do.** Call `feature_engineering` to merge **lookup tables** of cation and anion properties onto the main table.

**Why it matters.** A name like `Na+` is just text. Transport physics cares about things like ion size. Joining a property table turns chemistry knowledge into extra numeric **features** before any model is fit. This is *not* training, and it is *not* a physics simulator.

**What to look for.** `df_enriched` with extra columns (e.g. Stokes radius). Target columns should still be there, unchanged.

**Common pitfall.** Thinking “the model is trained now.” It is not. Ion names that do not match the lookup stay without those extra columns.

**Try it yourself.** After the cell runs, can you find a new column that was not in the raw table?


In [ ]:
from utl_feateng import*

In [ ]:
"""
Section 2: Feature engineering

This section defines a function that:
- Selects a physically meaningful subset of features from the raw table.
- Handles missing values by dropping high-missing or constant columns.
- Returns an ML-ready feature matrix X and target vector y.

The feature list is inspired by your RFE-selected features and descriptor columns
used in the model-training notebooks.
"""
import pandas as pd
import numpy as np


# Enrich the main dataframe with domain-informed features
# Using config-driven paths: passes config to feature_engineering function
df_enriched = feature_engineering(
    df_filtered,
    config=config,  # Pass config to load paths automatically
    verbose=True,
)

df_enriched

### The enriched table

You are looking at `df_enriched`: original recipe columns plus ion descriptors.

This tutorial only adds a **small** ion set (enough to show the idea). You could later add solvent properties or polymer descriptors the same way in `config.yaml`.

**What to look for.** Merge keys `S1_Cation` / `S1_Anion` produced real numbers for the solutes you care about.


### Optional: drop columns that never change

**What you will do.** `remove_low_variance_columns` removes features that are almost constant (except the targets).

**Why it matters.** A column that is the same in every row cannot help a prediction, but it still becomes extra dummy variables after one-hot encoding.

**What to look for.** A slightly narrower table and a printed variance threshold from `config.yaml`.

This step is **optional**. If you use another feature-selection method later, you might skip it. Keep it on for the default path.


In [ ]:
# After enrichment, we have 30+ features. We can apply the same variance and correlation filtering as before to get a more manageable set for modeling.
# low_variance_threshold lives under features: (not feature_engineering)
_features_cfg = config.get("features", {}) or {}
_fe_cfg = config.get("feature_engineering", {}) or {}
variance_threshold = _features_cfg.get(
    "low_variance_threshold",
    _fe_cfg.get("low_variance_threshold", 1e-9),
)

df_enriched_filtered, _ = remove_low_variance_columns(
    df_enriched, 
    exclude_columns=candidate_targets,
    variance_threshold=variance_threshold
)

print(f"Applied variance threshold: {variance_threshold}")

# Have a look at the filtered enriched dataframe before modeling
df_enriched_filtered


### Why we still keep the interesting knobs

Treatment temperatures that never vary in *this* extract can go. Monomer identity, additive amount, and polymerization time should **stay** — those are the levers a lab can change, and Section 6 will only be allowed to search a subset of them.

**Next.** Plots, then training. Modeling will again read `features:` from `config.yaml`.


## Section 3. Exploratory plots (before training)

**What you will do.** Plot how the targets are spread, then a **Pearson correlation** heatmap of numeric columns.

**Why it matters.** If almost all permeability values sit in a tiny band, a “good” score can be cheap. If two columns are nearly copies, the model may pick an arbitrary one. These plots are **quality control**, not the scientific conclusion.

**Words**

- **Pearson correlation:** a number from −1 to 1 for a *straight-line* relationship between two numeric columns. It ignores monomer *names* (those are categories).

**What to look for.** `target_distributions.png` (spread, skew, outliers) and `pearson_heatmap.png`. On Colab, files also appear under `outputs/figures/` in the repo folder.

**Common pitfall.** Dropping a chemically important knob only because the linear correlation is small. A tree model can still use nonlinear effects.

**Try it yourself.** Predict: will monomer *name* appear in the Pearson heatmap? (It should not.)


In [ ]:
from utl_eda import *
from utl_fileIO import file_io

fig, axes = plot_target_distributions(
    df_enriched_filtered,
    candidate_targets,
    fig_size=FIG_SIZE,
)

file_io("save", "figure", FIGURES_DIR, "target_distributions.png", data=fig)
print(f"Saved target distributions to {FIGURES_DIR}")

In [ ]:
# Compute and plot the Pearson correlation heatmap for the numeric features in the enriched dataframe, using different thresholds to focus on stronger correlations.

from utl_fileIO import file_io

df_enriched_numeric = df_enriched_filtered.select_dtypes(include=[np.number]) # Select only numeric columns for correlation analysis

# Use thresholds from config
correlation_thresholds = config["data_analysis"]["correlation_thresholds"]

for threshold in correlation_thresholds:
    corr_matrix, _heatmap_fig = compute_and_plot_pearson_heatmap(df_enriched_numeric, threshold=threshold, fig_size=[1.5 * x for x in FIG_SIZE])
    heatmap_fig = plt.gcf()
    file_io("save", "figure", FIGURES_DIR, f"pearson_heatmap_threshold_{threshold}.png", data=heatmap_fig)

print(f"Generated heatmaps with thresholds: {correlation_thresholds}")


### Reading the heatmap

Strong correlation in a literature table can mean “papers tend to report these two settings together,” not “A causes B.” Weak correlation does not mean “this variable is useless.”

**Next idea (Section 4).** We use **ExtraTrees**, an ensemble of decision trees, because membrane KPIs often depend on *combinations* (monomer × time × additive) that a single straight line misses.


In [ ]:
# Load modeling features from config.yaml to keep training inputs centralized and consistent.
import utl_config_loader as cfg_loader

base_features = cfg_loader.resolve_features_to_model(config, df_enriched_filtered)

if not base_features:
    raise ValueError(
        "No usable features found. With features.mode='manual', add your feature "
        "column names to features.include in config.yaml; with 'auto', ensure the "
        "dataset has usable non-target columns."
    )

# Optional visibility check against the enriched dataframe at this stage
missing_base_features = [f for f in base_features if f not in df_enriched_filtered.columns]
if missing_base_features:
    print("Features listed in config but missing from df_enriched_filtered:", missing_base_features)

print(f"Using {len(base_features)} base features from config.yaml")
for i, feature_name in enumerate(base_features, start=1):
    print(f"  {i}. {feature_name}")

In [ ]:
from sklearn.base import clone
from utl_model_arch import PipelineConfig, build_training_pipeline, get_hyperparameter_space, build_model_pipeline

modeling_cfg = config["modeling"]
pipeline_cfg_dict = modeling_cfg.get("pipeline", {})
hyperparameter_space = get_hyperparameter_space(config)

pipeline_template = build_training_pipeline(PipelineConfig(**pipeline_cfg_dict))
print("Pipeline template built via utl_model_arch.PipelineConfig")

print("Hyperparameter space keys:", list(hyperparameter_space.keys()))

## Section 4. Train ExtraTrees

**What you will do.** For each continuous target, fit a scikit-learn **pipeline** (preprocessing + ExtraTrees), score it, plot **parity**, and save a `.pkl` model.

**Why it matters.** We need a **surrogate**: a fast stand-in for “make the membrane and measure it.” SHAP and Ax will query this saved pipeline, so the train/test split has to be honest.

**Words**

- **Train set:** rows used to fit the pipeline (including how missing values are filled).
- **Test set:** rows the model must not have fit on — our estimate of performance on new rows.
- **One-hot encoding:** turning a category (e.g. monomer name) into 0/1 columns.
- **Leakage:** accidentally using test information during training (e.g. filling missing values with the *full* table mean).
- **R²:** fraction of variance explained (1 is perfect on that set; can be poor on test).
- **RMSE / MAE:** typical size of prediction errors (same units as the target). RMSE punishes large misses more.

**What to look for.** Printed train *and* test metrics; `parity_<target>.png`; files in `models/`. A much better train than test score usually means overfitting.

**Common pitfall.** `force_retrain=False` reuses an old pickle after you changed `config.yaml`. Delete `models/` or set `force_retrain=True`.

**Try it yourself.** If train R² = 0.99 and test R² = 0.2, is the model ready for inverse design? Why or why not?


In [ ]:
# evaluate_and_plot_parity dispatches on task: parity (regression)
# or confusion matrices (classification). Inverse design later is
# continuous/regression only.
import utl_modelling
from utl_fileIO import file_io

train_model = utl_modelling.train_model
evaluate_and_plot_parity = utl_modelling.evaluate_and_plot_parity

# Loop over candidate targets
for target in candidate_targets:
    print(f"\n=== Modeling target: {target} ===")

    # Clone to keep each target run isolated from fitted state.
    model_pipeline = clone(pipeline_template)

    pipe, X_train, X_test, y_train, y_test = train_model(
        df=df_enriched_filtered,
        target=target,
        pipeline=model_pipeline,
        config_path=config["_config_path"],
        force_retrain=False,
        param_dist_override=hyperparameter_space,
    )

    # Predict on train and test
    y_pred_train = pipe.predict(X_train)
    y_pred_test = pipe.predict(X_test)

    evaluate_and_plot_parity(
        target=target,
        y_train=y_train,
        y_pred_train=y_pred_train,
        y_test=y_test,
        y_pred_test=y_pred_test,
        fig_size=FIG_SIZE,
    )

    safe_target = "".join(ch if ch.isalnum() else "_" for ch in str(target)).strip("_")
    parity_fig = plt.gcf()
    file_io("save", "figure", FIGURES_DIR, f"parity_{safe_target}.png", data=parity_fig)

print(f"Completed modeling for {len(candidate_targets)} targets")


### How to read a parity plot

Each point is one experiment: measured KPI on one axis, predicted KPI on the other. The diagonal means “prediction = measurement.”

**What to look for.** Points near the diagonal **across the whole range**, not only in the middle. A tight train cloud and a scattered test cloud is overfitting. A bend at high permeability means the model is weak in that region — Ax suggestions there would be shaky.

Classification problems use a **confusion matrix** instead; this tutorial’s KPIs are continuous.

**Try it yourself.** Point to where on the plot you would look to see “the model always under-predicts high rejection.”


## Section 5. SHAP — who is the model listening to?

**What you will do.** Load the **saved** ExtraTrees pipelines and draw SHAP beeswarm and bar plots.

**Why it matters.** A test score says *whether* predictions match. SHAP says *which input columns* pushed the prediction up or down in this fitted model. That is explanation of the **surrogate**, not a proof of membrane physics, and not causation.

**SHAP** (SHapley Additive exPlanations) assigns each feature a contribution for each sample, using ideas from cooperative game theory. You do not need the math to read the plots.

**What to look for.** `shap_summary_*.png` in `outputs/figures/`. Top rows = largest average effect.

**Common pitfall.** “SHAP says additive loading causes rejection.” Safer: “In this model, higher additive loading was associated with higher/lower predicted rejection.” Needs the `shap` package (`pip install -e ".[shap]"`). If SHAP fails, you can still run Section 6 using lab knowledge to choose search variables.


In [ ]:
from utl_shap import*

### Beeswarm vs bar

- **Beeswarm:** each dot is one sample. Color = high vs low feature value; left/right = decrease vs increase in the prediction. Use this for **direction**.
- **Bar:** mean |SHAP| — a simple **ranking**.

**Try it yourself.** Circle one feature you could actually change in a lab. That is a candidate for the Ax **search space** next. Features you cannot control belong in **constants**.


In [ ]:
import utl_shap
from utl_fileIO import file_io

# Loop over candidate targets to compute and plot SHAP values for model interpretation
for target in candidate_targets:
    shap_values, feature_names, summary_fig, bar_fig = utl_shap.shap_explain_target(
        target=target,
        df_enriched=df_enriched,
        features_to_model=base_features,
        config=config,
        max_samples=1000,
    )

    safe_target = "".join(ch if ch.isalnum() else "_" for ch in str(target)).strip("_")
    file_io("save", "figure", FIGURES_DIR, f"shap_summary_{safe_target}.png", data=summary_fig)
    file_io("save", "figure", FIGURES_DIR, f"shap_importance_{safe_target}.png", data=bar_fig)

print(f"Completed SHAP analysis for {len(candidate_targets)} targets")

### From plots to a hypothesis

If monomer, additive, time, or ion size rank high *and* the direction matches what you know about IP membranes, the model is at least consistent with chemistry. If the top feature is a paper ID or a column that never varies in the lab, something is wrong with the feature list.

Section 6 should only vary a **short** list of actionable knobs. If that list fights both SHAP and common sense, fix the YAML before trusting Ax.


## Section 6. Inverse design with Ax

**What you will do.** Treat ExtraTrees as a cheap “test the recipe” function and let **Ax** suggest fabrication settings that improve a combined score of permeability and rejection.

**Why it matters.** Measuring every possible monomer / time / additive mix in the lab is slow. Inverse design = *search*, not *proof*. The optimizer never makes a real membrane.

**Analogy.** You have a flight simulator (the model) and you try cockpit settings (the search space) to improve a score. Useful for a shortlist; you still fly the real plane (the experiment).

**Ax** is an open-source Bayesian optimization toolkit. **Bayesian optimization** proposes the next experiment using a model of “where good scores might be,” balancing new regions vs known good ones.

**What to look for.** This section **requires** Ax (`pip install -e ".[opt]"`). The first install can take several minutes. You should see completed trials, printed recipes, and later `inverse_design_summary.png`. On Colab those files are under `/content/ML-MEM-Review/outputs/figures/` (paths from §0).

**Common pitfall.** Running §6 on **classification** labels, or without the `.pkl` files from §4.

**Try it yourself.** Before running: if we weight rejection much more than permeability in the score, should the suggested recipes look more “tight” (selective) or more “open” (permeable)?


### Search space vs constants

**Search space:** knobs Ax may change (ranges or choices), e.g. monomer name, masses, polymerization time — listed in `optimization.search_space`.

**Constants:** everything the model still needs but you refuse to vary (`constant_parameters`), e.g. a solvent you will not switch.

**Hard rule.** Searchable + constant columns together must match what ExtraTrees was trained on. If the oracle row is missing a feature, prediction is undefined.

Keep the space **small** so suggestions stay lab-plausible.

**What to look for.** The next cell prints how many Ax parameters were built. Check that list against your protocol.


In [ ]:
if Client is None or RangeParameterConfig is None or ChoiceParameterConfig is None:
    import subprocess
    spec = f"{PROJECT_ROOT}[opt]"
    print(f"Ax not found; installing {spec}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", spec])
    from ax.api.client import Client
    from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig
import utl_Ax
from utl_Ax import *


In [ ]:
# Define a compact search space of IP variables and conditions using config values.
# Tutorial guidance: start narrow (chemically plausible bounds), then expand after validation.

from utl_Ax import build_ax_search_space

opt_cfg = config["optimization"]

search_space_parameters = build_ax_search_space(
    opt_cfg=opt_cfg,
    choice_param_cls=ChoiceParameterConfig,
    range_param_cls=RangeParameterConfig,
)

print(f"Built search space with {len(search_space_parameters)} parameters from config")

### The oracle (the model as a fake experiment)

Each Ax trial calls `oracle_from_params`:

1. Merge the proposed knobs with constants.
2. Build a one-row table and run the **same** `feature_engineering` as Section 2.
3. Predict KPIs with the saved pipelines.
4. Combine them into the scalar score from `config.yaml`.

**Common pitfall.** Missing `models/*.pkl` (go back to §4) or a YAML name that is not a training feature.

You do not call the oracle yourself; `run_ax_optimization` does it every trial.


### Combining two goals into one number

We usually want **both** high water permeability **and** high rejection. This notebook uses **scalarization**: scale each KPI toward 0–1, then combine with an equation in `optimization.scalarized_objective.equation` (e.g. a weighted sum).

**What to look for.** The next cell prints weights from `optimization.objectives`. Changing 0.3 vs 0.7 **changes what “best” means**.

### Optional deeper dive

The package can also run **multiobjective** (Pareto) mode. That is off by default here. Do not treat the scalarized winner as “the” optimal membrane — it is optimal *for this equation and these bounds*.


In [ ]:
# Build objective weight map used for scalarized plots and summaries.
opt_cfg = config["optimization"]

objective_weights = {}
for obj_cfg in opt_cfg["objectives"]:
    objective_name = obj_cfg["name"]
    weight = float(obj_cfg.get("weight", 1.0))
    objective_weights[objective_name] = weight

print("Loaded objective weights from config:")
for key, weight in objective_weights.items():
    print(f"  - {key}: {weight}")

### The optimization loop

**What you will do.** Run `run_ax_optimization`: a few random (Sobol) starts, then Bayesian trials. `n_trials` in YAML is **small** so the demo finishes. This cell can take several minutes.

**What to look for.** Logs ending with `Optimization completed: N trials`. A checkpoint is saved under `models/` (`client_cache_file`).

**Common pitfall.** No Ax in this kernel. Re-run bootstrap or `pip install -e ".[opt]"`, then this cell.

Treat the result as a **ranked shortlist of experiments**, not a new membrane.


In [ ]:
print(f"Optimization mode: {opt_cfg['mode']}")

client, results = run_ax_optimization(
    search_space_parameters=search_space_parameters,
    oracle_from_params=oracle_from_params,
    sobol_init=opt_cfg["sobol_init"],
    n_trials=opt_cfg["n_trials"],
    force_rebuild=opt_cfg.get("force_rebuild", False),
        random_seed=config.get("modeling", {}).get("random_state"),
    save_every_trials=opt_cfg["save_checkpoint_every"],
    client_path=os.path.join(MODEL_PATH, opt_cfg["client_cache_file"]),
    config=config,
)

print(f"Optimization completed: {len(results)} trials")

### Reading the suggested recipes

**What you will do.** Print the top recommendations (`get_optimization_recommendations`).

“Best” means: best **scalarized score** inside **this** search box, with **these** weights and constants — according to **this** ExtraTrees fit.

**What to look for.** Recipes that a chemist would actually mix. If Ax proposes nonsense, the surrogate, bounds, or equation is wrong.

**Try it yourself.** Would you run the top recipe in the lab this week? What measurement would make you trust or reject the surrogate?


In [ ]:
# Retrieve and inspect optimization recommendations from Ax.
# Works for both scalarized (single best point) and multiobjective (Pareto frontier).

import warnings
from numbers import Number

recommendations = get_optimization_recommendations(
    client=client,
    optimization_mode=opt_cfg["mode"],
    top_k=5,
)

# Tutorial next step: validate top recommendation(s) experimentally (or with held-out data),
# then append outcomes and rerun BO for closed-loop improvement.

### Pictures of the search

**What you will do.** Save `inverse_design_summary.png` (how scores moved over trials) and optional response-map PNGs.

**What to look for.** Whether the combined score improved after the random start. Response maps are a smooth cartoon of **this short run**, not a map of nature. If a map is skipped (too few distinct points), the summary figure is still enough.

Files: the figures directory printed in Section 0 (Colab: under `/content/ML-MEM-Review/...`).


In [ ]:
import utl_Ax

from utl_config_loader import load_config
from utl_Ax import plot_ax_results
from utl_fileIO import file_io

config = load_config(project_root=PROJECT_ROOT)
trial_df = results_to_dataframe(results)

viz = plot_ax_results(trial_df=trial_df, config=config, client=client)

summary_figure = viz.get("summary", {}).get("figure") if isinstance(viz.get("summary"), dict) else None
if summary_figure is None and viz.get("figures"):
    summary_figure = viz["figures"].get("summary_figure")
if summary_figure is not None:
    file_io("save", "figure", FIGURES_DIR, "inverse_design_summary.png", data=summary_figure)

for index, response_map in enumerate(viz.get("response_maps", []), start=1):
    if response_map.get("contour_figure") is not None:
        file_io(
            "save",
            "figure",
            FIGURES_DIR,
            f"response_map_{index}_contour.png",
            data=response_map["contour_figure"],
        )
    if response_map.get("surface_figure") is not None:
        file_io(
            "save",
            "figure",
            FIGURES_DIR,
            f"response_map_{index}_surface.png",
            data=response_map["surface_figure"],
        )

from pathlib import Path
from IPython.display import display, Markdown, Image
import matplotlib.pyplot as plt

plt.close("all")
for name in [
    "inverse_design_summary.png",
    "response_map_1_contour.png",
    "response_map_1_surface.png",
    "response_map_2_contour.png",
    "response_map_2_surface.png",
]:
    path = Path(FIGURES_DIR) / name
    if path.is_file():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(path)))


## Section 7. Recap, troubleshooting, next steps

You completed one loop: **table → extra chemistry columns → ExtraTrees → SHAP → Ax shortlist**.

### Key concepts

- Features vs targets; train vs test; correlation vs causation.
- The model is a surrogate. SHAP explains the surrogate. Ax searches the surrogate.
- Inverse design is for **continuous** KPIs in this package.

### Troubleshooting

| Problem | What to try |
|---------|-------------|
| `config.yaml` / `utl_*` not found | Run the Colab bootstrap, then §0, in order. |
| SHAP import error | `pip install -e ".[shap]"`; §6 can continue without SHAP. |
| Ax / optimize error | `pip install -e ".[opt]"` in **this** kernel; confirm §4 wrote `.pkl` files. |
| Old figures / old model | `force_retrain=True` or delete `models/`. |
| Colab clone looks old | Re-run bootstrap after a GitHub push. |

### Practice (pick one)

1. Change `data.targets` in `config.yaml` to another continuous column and re-run §4–§5.
2. Set `modeling.model_selection` to `randomforest` and compare test RMSE.
3. Change the 0.3 / 0.7 weights in the scalarized equation and see whether Ax’s top recipe moves.

None of these replace making the membrane. They help you choose **which** membrane to make next.
